# Initialize: dynamic Colonel Blotto benchmark
Defines the environment, the latent context, the defender's two-part representation, the stage-game solvers, the networks, the training updates, the baseline architectures, and the defender wrappers used by the other notebooks.

Conventions:
- **Game.** A defender and an attacker each allocate $B = 4$ troop units over $N = 5$ towns per round ($|A^1| = |A^2| = 70$) for $T = 50$ rounds. The state $x_k \in [0, 1]^N$ is the defender's per-town control.
- **Dynamics.** Town $i$ moves toward $f_i = \sigma(2.5\,(a_i - u_i))$ with mixture weight $\rho_i$: $x_{k+1,i} \sim \mathrm{Beta}(25\mu_i, 25(1-\mu_i))$, with $\mu_i = (1-\rho_i)\,x_{k,i} + \rho_i f_i$.
- **Payoff.** `reward` returns the **defender's** payoff, the value-weighted control $r_k = \langle w, x_{k+1}\rangle$. The critic learns the defender's payoff, and the defender is the row maximizer in the stage game; this is the paper's convention with the sign flipped.
- **Latent context** $\theta = (\rho, w, R)$, drawn once per episode by `Sample_Context`:
  - the responsiveness $\rho_i \in \{0.1, 0.9\}$, unknown to both players,
  - the attacker's valuations $w$ (three valued towns with weights 0.15, 0.30, and 0.55), and
  - its reachability mask $R$ (the three towns it may strike).

  The attacker knows its type $(w, R)$. Its admissible actions are the allocations that put no troops on unreachable towns.
- **Observations.** The defender sees only its own actions and a noisy reading of control, $y_k = \mathrm{clip}(x_k + \mathcal N(0, 0.05^2))$.
- **Representation** $z_k = [z^S_k, z^\Theta_k]$:
  - the state component $z^S$ is a GRU cell over $(y_k, a^1_{k-1})$ (`StateGRU`), trained through the defender's distillation loss;
  - the context component $z^\Theta$ is a Transformer over the history rows $[a^1_{k-1}, y_k]$ (`TSTransformer`).
- **Attacker's private state** $m^2_k$: a per-town posterior over $\rho_i \in \{0.1, 0.9\}$, marginalizing the defender's unseen allocation under a uniform prior (`m2_update`).
- **Encoder objective**, Eq. (3): $\mathcal L_{\mathrm{enc}} = \lambda_r \mathcal L_r + \lambda_p \mathcal L_p + \lambda_\Theta \mathcal L_{\mathrm{SI}}$ (`Train_Encoder`).
- **Game step.** The critic is trained by TD regression onto the value of the next stage game, restricted to the attacker's admissible actions. Because the game is 70 by up to 70, the value is computed by batched regret matching (`batched_game_values`, within about 0.05% of the exact LP), and the actors' equilibrium targets by the exact LP. The state and context components are recomputed with the current encoders at every update.

In [ ]:
import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import linprog
from scipy.stats import beta as beta_dist
from itertools import combinations_with_replacement
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.distributions import Categorical

torch.backends.mha.set_fastpath_enabled(False)      # keeps the padding-mask path identical in train and eval
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Benchmark definition: 5 towns, budget 4, 50 rounds

In [ ]:
N = 5                                   # towns
B = 4                                   # troop budget per round
actions = []
for combo in combinations_with_replacement(range(N), B):
    vec = np.zeros(N, dtype=int)
    for town in combo:
        vec[town] += 1
    actions.append(vec)
actions = np.array(actions)             # (70, N): every allocation of B troops over N towns
action_to_idx = {tuple(vec): i for i, vec in enumerate(actions)}
n_a = len(actions)
n_u = len(actions)

Len_Theta = 3 * N                       # theta = [rho (N), w (N), R (N)]
Embedding_Size = 64
Z_S_Size = 32
max_steps = 50                          # rounds per episode, also the context window

RHO_VALUES = np.array([0.1, 0.9])       # responsiveness
W_VALUES = [0.15, 0.30, 0.55]           # weights of the three valued towns
K_SLOPE = 2.5                           # steepness of the contest function
BETA_CONC = 25.0                        # concentration of the Beta transition
OBS_NOISE = 0.05
P_AI = np.array([(actions[:, 0] == k).mean() for k in range(B + 1)])   # troops on one town under a uniform allocation


def legal_actions(Reachable):
    # Attacker allocations that put no troops on unreachable towns
    return np.where(actions[:, np.where(Reachable == 0)[0]].sum(axis=1) == 0)[0]


def Sample_Context():
    # Draw the episode's latent context theta = (rho, w, R)
    Theta = np.zeros(Len_Theta)
    Theta[0:N] = np.random.choice(RHO_VALUES, size=N)
    Reachable = np.zeros(N)
    Reachable[np.random.choice(N, size=3, replace=False)] = 1.0
    valued = np.random.choice(N, size=3, replace=False)
    Weights = np.zeros(N)
    Weights[valued] = np.random.permutation(W_VALUES)
    Type_Theta = np.concatenate([Weights, Reachable])
    Theta[N:Len_Theta] = Type_Theta
    return Theta, Type_Theta, Weights, Reachable, legal_actions(Reachable)


def sample_config():
    # Context and initial state, drawn in the same order as main.ipynb
    Theta, Type_Theta, Weights, Reachable, legal = Sample_Context()
    x0 = np.random.beta(2.0, 2.0, size=N)
    return Theta, Weights, Reachable, x0, legal

## Dynamics, observations, reward, attacker's private state

In [ ]:
def next_state(x, a, u, Theta):
    # a, u: troop vectors of the defender and the attacker
    rho = Theta[0:N]
    f = 1.0 / (1.0 + np.exp(-K_SLOPE * (a - u)))
    mu = np.clip((1.0 - rho) * x + rho * f, 0.0001, 0.9999)
    return np.random.beta(BETA_CONC * mu, BETA_CONC * (1.0 - mu))
#################################################################################################################
def Observation(next_x):
    return np.clip(next_x + np.random.normal(0, OBS_NOISE, size=N), 0.0, 1.0)
#################################################################################################################
def reward(x, Theta):
    # Defender's payoff: value-weighted control
    return float(np.dot(Theta[N:2*N], x))
#################################################################################################################
def m2_update(m2, x, u, next_x):
    # Attacker's posterior over rho_i in {0.1, 0.9} per town, marginalizing the defender's unseen troops
    m2 = m2.reshape(N, 2).copy()
    nx = np.clip(next_x, 1e-4, 1 - 1e-4)
    for i in range(N):
        lik = np.zeros(2)
        for l, rho in enumerate(RHO_VALUES):
            acc = 0.0
            for ai in range(B + 1):
                f = 1.0 / (1.0 + np.exp(-K_SLOPE * (ai - u[i])))
                mu = np.clip((1.0 - rho) * x[i] + rho * f, 1e-4, 1 - 1e-4)
                acc += P_AI[ai] * beta_dist.pdf(nx[i], BETA_CONC * mu, BETA_CONC * (1 - mu))
            lik[l] = acc
        post = m2[i] * np.clip(lik, 1e-30, 1e30)
        s = post.sum()
        m2[i] = post / s if (np.isfinite(s) and s > 1e-300) else np.full(2, 0.5)
        if not np.all(np.isfinite(m2[i])):
            m2[i] = np.full(2, 0.5)
    return m2.reshape(N * 2)
#################################################################################################################
def Context_Row(a_vec, y):
    # One history row for the encoders: [defender's troops, observed control]
    row = np.zeros(N * 2, dtype=np.float32)
    row[0:N] = a_vec
    row[N:2*N] = y
    return row

def Context_Mask(num_rows):
    mask = torch.zeros((1, max_steps), dtype=torch.bool, device=device)
    mask[0, :max(1, min(num_rows, max_steps))] = True
    return mask

## Stage-game solvers

In [ ]:
def MatrixValue(M):
    # Exact minimax by linear programming. Row player maximizes, column player minimizes.
    # Returns (value, row strategy p, column strategy q).
    M = np.asarray(M, dtype=float)
    if M.ndim != 2:
        raise ValueError("M must be a 2D array.")
    m, n = M.shape
    if m == 0 or n == 0 or not np.isfinite(M).all():
        return float(np.nanmean(M)), np.ones(m)/m if m else np.array([]), np.ones(n)/n if n else np.array([])
    # normalization
    center = float(M.mean())
    A = M - center
    scale = float(np.max(np.abs(A)))
    if scale == 0.0:
        return center, np.ones(m)/m, np.ones(n)/n
    A /= scale
    c1 = np.zeros(m + 1); c1[0] = -1.0
    A_ub1 = np.zeros((n + m, m + 1))
    b_ub1 = np.zeros(n + m)
    A_ub1[:n, 0] =  1.0
    A_ub1[:n, 1:] = -A.T
    A_ub1[n:, 1:] = -np.eye(m)
    A_eq1 = np.ones((1, m + 1)); A_eq1[0, 0] = 0.0
    b_eq1 = np.array([1.0])
    opt1 = linprog(c=c1, A_ub=A_ub1, b_ub=b_ub1, A_eq=A_eq1, b_eq=b_eq1, method="highs", options={"presolve": True})
    v_row = None; p = None
    if opt1.success and opt1.x is not None:
        v_row = float(opt1.x[0])
        p = opt1.x[1:]
    c2 = np.zeros(n + 1); c2[0] = 1.0
    A_ub2 = np.zeros((m + n, n + 1))
    b_ub2 = np.zeros(m + n)
    A_ub2[:m, 0]  = -1.0
    A_ub2[:m, 1:] =  A
    A_ub2[m:, 1:] = -np.eye(n)
    A_eq2 = np.ones((1, n + 1)); A_eq2[0, 0] = 0.0
    b_eq2 = np.array([1.0])
    opt2 = linprog(c=c2, A_ub=A_ub2, b_ub=b_ub2, A_eq=A_eq2, b_eq=b_eq2, method="highs", options={"presolve": True})
    v_col = None; q = None
    if opt2.success and opt2.x is not None:
        v_col = float(opt2.x[0])
        q = opt2.x[1:]
    if p is None and q is None:
        return center, np.ones(m)/m, np.ones(n)/n
    if p is None:
        p = np.ones(m)/m
        v_norm = v_col
    elif q is None:
        q = np.ones(n)/n
        v_norm = v_row
    else:
        v_norm = 0.5 * (v_row + v_col)
    p = np.clip(p, 0.0, 1.0); ps = p.sum(); p = p/ps if ps > 0 else np.ones(m)/m
    q = np.clip(q, 0.0, 1.0); qs = q.sum(); q = q/qs if qs > 0 else np.ones(n)/n
    value = center + scale * float(v_norm)
    return value, p, q
#################################################################################################################
def batched_game_values(Q, col_mask, iters=200):
    # Values of a batch of stage games by regret matching+ with linearly averaged strategies.
    # Q: (batch, n_a, n_u) defender payoff; the row player maximizes; the column player minimizes over col_mask.
    Bsz, m, n = Q.shape
    cm = col_mask.float()
    Rp = torch.zeros(Bsz, m, device=Q.device); Rq = torch.zeros(Bsz, n, device=Q.device)
    p_sum = torch.zeros_like(Rp); q_sum = torch.zeros_like(Rq)
    for it in range(1, iters + 1):
        p = Rp.clamp(min=0); s = p.sum(1, keepdim=True)
        p = torch.where(s > 0, p / s.clamp(min=1e-12), torch.full_like(p, 1.0 / m))
        q = Rq.clamp(min=0) * cm; s = q.sum(1, keepdim=True)
        q = torch.where(s > 0, q / s.clamp(min=1e-12), cm / cm.sum(1, keepdim=True))
        row_pay = torch.bmm(Q, q.unsqueeze(2)).squeeze(2)
        col_pay = torch.bmm(p.unsqueeze(1), Q).squeeze(1)
        v = (p * row_pay).sum(1, keepdim=True)
        Rp = (Rp + row_pay - v).clamp(min=0)
        Rq = ((Rq + v - col_pay) * cm).clamp(min=0)
        p_sum += it * p; q_sum += it * q
    p_bar = p_sum / p_sum.sum(1, keepdim=True)
    q_bar = q_sum / q_sum.sum(1, keepdim=True)
    return torch.einsum('bm,bmn,bn->b', p_bar, Q, q_bar)
#################################################################################################################
ACTIONS_T = torch.tensor(actions, dtype=torch.float32, device=device)

def legal_mask_from_theta(th):
    # (batch, n_u) mask of the attacker's admissible actions, from theta's reachability block
    unreach = 1.0 - th[:, 2*N:3*N]
    return (unreach @ ACTIONS_T.t()) == 0

## Replay buffer and networks

In [ ]:
class ReplayMemory(object):
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)

    def push(self, args):
        self.memory.append(args)

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)
#################################################################################################################
class StateGRU(nn.Module):
    # State component z^S: GRU cell over (y_k, a_{k-1})
    def __init__(self):
        super().__init__()
        self.hidden = Z_S_Size
        self.n = N
        self.cell = nn.GRUCell(2 * self.n, self.hidden)

    def forward(self, z_s, y, a_prev):
        inp = torch.cat([y, a_prev.float()], dim=-1)
        return self.cell(inp, z_s)
#################################################################################################################
class TSTransformer(nn.Module):
    # Context component z^Theta: Transformer with learned positional embeddings, masked mean pooling, feedforward head
    def __init__(self):
        super(TSTransformer, self).__init__()
        self.input_projection = nn.Linear(N*2, 128)
        self.position_embeddings = nn.Embedding(max_steps, 128)
        encoder_layers = TransformerEncoderLayer(d_model=128, nhead=4, batch_first=True, dropout=0.1)
        self.transformer_encoder = TransformerEncoder(encoder_layers, num_layers=3, enable_nested_tensor=False)
        self.fc1 = nn.Linear(128, 128)
        self.norm1 = nn.LayerNorm(128)
        self.fc2 = nn.Linear(128, Embedding_Size)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, mask):
        x = self.input_projection(x)
        batch_size, seq_length, _ = x.shape
        position_ids = torch.arange(seq_length, device=x.device).unsqueeze(0).expand(batch_size, -1)
        x = x + self.position_embeddings(position_ids)
        key_padding_mask = ~mask.bool()
        x = self.transformer_encoder(x, src_key_padding_mask=key_padding_mask)
        mask = mask.unsqueeze(-1)
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        x = self.fc1(x)
        x = self.norm1(x)
        x = self.activation(x)
        x = self.dropout(x)
        return self.fc2(x)
#################################################################################################################
class Critic(nn.Module):
    # Centralized Q(z^S, z^Theta, state, m^2, theta_type, theta) over all joint actions, the defender's payoff
    def __init__(self):
        super().__init__()
        self.enc_zs1    = nn.Linear(Z_S_Size, 128)
        self.enc_zs2    = nn.Linear(128, 2)
        self.enc_embd1  = nn.Linear(Embedding_Size, 128)
        self.enc_embd2  = nn.Linear(128, 2)
        self.enc_state1 = nn.Linear(N, 128)
        self.enc_state2 = nn.Linear(128, 128)
        self.enc_m2_1   = nn.Linear(N*2, 128)
        self.enc_m2_2   = nn.Linear(128, 2)
        self.enc_type1  = nn.Linear(N*2, 64)
        self.enc_type2  = nn.Linear(64, 2)
        self.enc_theta1 = nn.Linear(Len_Theta, 64)
        self.enc_theta2 = nn.Linear(64, 128)

        self.ln_fuse = nn.LayerNorm(128 * 2 + 4 * 2)
        self.fuse1   = nn.Linear(128 * 2 + 4 * 2, 512)
        self.fuse2   = nn.Linear(512, 512)
        self.out     = nn.Linear(512, n_a * n_u)

    def forward(self, z_s, Embeding, x, m2, type_theta, theta):
        z_b = F.relu(self.enc_zs2(F.relu(self.enc_zs1(z_s.detach()))))
        z_c = F.relu(self.enc_embd2(F.relu(self.enc_embd1(Embeding))))
        z_x = F.relu(self.enc_state2(F.relu(self.enc_state1(x))))
        z_m = F.relu(self.enc_m2_2(F.relu(self.enc_m2_1(m2))))
        z_y = F.relu(self.enc_type2(F.relu(self.enc_type1(type_theta))))
        z_t = F.relu(self.enc_theta2(F.relu(self.enc_theta1(theta))))
        e = torch.cat([z_b, z_c, z_x, z_m, z_y, z_t], dim=-1)
        e = self.ln_fuse(e)
        e = F.relu(self.fuse1(e))
        e = F.relu(self.fuse2(e))
        return self.out(e)
#################################################################################################################
class Player2Actor(nn.Module):
    # beta(u | state, m^2, theta_type), masked to the admissible allocations
    def __init__(self):
        super().__init__()
        self.enc_x1   = nn.Linear(N, 128)
        self.enc_x2   = nn.Linear(128, 128)
        self.enc_m2_1 = nn.Linear(N*2, 128)
        self.enc_m2_2 = nn.Linear(128, 128)
        self.enc_ty1  = nn.Linear(N*2, 64)
        self.enc_ty2  = nn.Linear(64, 128)
        self.fuse1 = nn.Linear(384, 256)
        self.ln    = nn.LayerNorm(256)
        self.fuse2 = nn.Linear(256, 128)
        self.head  = nn.Linear(128, n_u)
        self.register_buffer('actions_t', torch.tensor(actions, dtype=torch.float32))

    def forward(self, x, m2, type_theta):
        zx = F.relu(self.enc_x2(F.relu(self.enc_x1(x))))
        zm = F.relu(self.enc_m2_2(F.relu(self.enc_m2_1(m2))))
        zt = F.relu(self.enc_ty2(F.relu(self.enc_ty1(type_theta))))
        h = torch.cat([zx, zm, zt], dim=-1)
        h = self.ln(F.relu(self.fuse1(h)))
        h = F.relu(self.fuse2(h))
        logits = self.head(h)
        Reachable = type_theta[..., N:2*N]
        legal = (self.actions_t * (1.0 - Reachable).unsqueeze(-2)).sum(-1) == 0
        probs = F.softmax(logits.masked_fill(~legal, -1e9), dim=-1) * legal
        return probs / probs.sum(dim=-1, keepdim=True).clamp(min=1e-12)
#################################################################################################################
class Player1Actor(nn.Module):
    # pi^1(a | z), with z = [z^S, z^Theta]; returns logits
    def __init__(self):
        super().__init__()
        self.enc_zs1  = nn.Linear(Z_S_Size, 128)
        self.enc_zs2  = nn.Linear(128, 128)
        self.enc_emb1 = nn.Linear(Embedding_Size, 128)
        self.enc_emb2 = nn.Linear(128, 128)
        self.fuse1 = nn.Linear(256, 256)
        self.ln    = nn.LayerNorm(256)
        self.fuse2 = nn.Linear(256, 128)
        self.head  = nn.Linear(128, n_a)

    def forward(self, z_s, Embeding):
        zb = F.relu(self.enc_zs2(F.relu(self.enc_zs1(z_s))))
        zc = F.relu(self.enc_emb2(F.relu(self.enc_emb1(Embeding))))
        h = torch.cat([zb, zc], dim=-1)
        h = self.ln(F.relu(self.fuse1(h)))
        h = F.relu(self.fuse2(h))
        return self.head(h)

## Representation step: encoder objective, Eq. (3)
$\mathcal L_{\mathrm{enc}} = \lambda_r \mathcal L_r + \lambda_p \mathcal L_p + \lambda_\Theta \mathcal L_{\mathrm{SI}}$:
- $\mathcal L_r$: regression of the stage reward on the representation, through a reward head.
- $\mathcal L_p$: prediction of the next representation from $(z_k, a^1_k, y_{k+1})$ through a latent transition model and a prediction head, against gradient-free exponential-moving-average copies of the encoder and projection. `F.mse_loss` averages over the 128 projection dimensions, so it is the squared norm in Eq. (3) divided by 128; the constant is absorbed into $\lambda_p$.
- $\mathcal L_{\mathrm{SI}}$: Soft-InfoNCE on the $\ell_2$-normalized projection, $\mathcal L_i = -\sum_{j\ne i} w_{ij}\log p_{ij}$, with $p_{ij}$ a softmax of $g_i^\top g_j/T$ over $k \ne i$ and graded targets $w_{ij} = s_{ij}/\sum_{\ell\ne i}s_{i\ell}$, $s_{ij} = \exp(-\tilde d_\Theta(\theta_i,\theta_j)/\sigma_\Theta)$.

The computable proxy is $\tilde d_\Theta = d_R + d_w$ with $\sigma_\Theta = 1$, over the attacker's type:
- $d_R = -\log J(R_i, R_j)$, with $J$ the Jaccard similarity of the reachable sets, and
- $d_w = -\log \cos(w_i, w_j)$, the cosine similarity of the valuations.

Any two three-town sets overlap, so both similarities are positive. $s_{ij}$ is their product (`Context_Similarity`). The responsiveness $\rho$ gets weight $\lambda_\rho = 0$.

In [ ]:
def Context_Similarity(th):
    # s_ij = exp(-d_tilde(theta_i, theta_j)) = J(R_i, R_j) * cos(w_i, w_j)
    w = th[:, N:2*N]
    R = th[:, 2*N:3*N]
    inter = R @ R.t()
    union = R.sum(1, keepdim=True) + R.sum(1).unsqueeze(0) - inter
    reach_sim = inter / union.clamp(min=1e-9)
    wn = F.normalize(w, p=2, dim=1)
    w_sim = (wn @ wn.t()).clamp(min=0.0)
    return reach_sim * w_sim
#################################################################################################################
def ema_update(tau=0.995):
    # Gradient-free targets for the next-representation loss
    for po, pt in zip(Context_Model.parameters(), target_model.parameters()):
        pt.data.lerp_(po.data, 1.0 - tau)
    for po, pt in zip(proj_online.parameters(), proj_target.parameters()):
        pt.data.lerp_(po.data, 1.0 - tau)
    for bo, bt in zip(Context_Model.buffers(), target_model.buffers()):
        bt.data.copy_(bo.data)
    for bo, bt in zip(proj_online.buffers(), proj_target.buffers()):
        bt.data.copy_(bo.data)
#################################################################################################################
def Train_Encoder():
    if len(context_memory) < CONTEXT_BATCH:
        return 0.0
    batch = context_memory.sample(CONTEXT_BATCH)
    (ctx, msk, nctx, nmsk, a, y, r, th) = map(list, zip(*batch))
    ctx  = torch.cat(ctx,  dim=0).to(device)
    msk  = torch.cat(msk,  dim=0).to(device)
    nctx = torch.cat(nctx, dim=0).to(device)
    nmsk = torch.cat(nmsk, dim=0).to(device)
    a  = torch.stack(a).to(device)
    y  = torch.stack(y).to(device).float()
    r  = torch.stack(r).to(device).float()
    th = torch.stack(th).to(device).float()
    Bsz = ctx.shape[0]

    Context_Model.train(); trans_pred.train(); reward_head.train()
    proj_online.train();  pred_online.train()
    z = Context_Model(ctx, msk)

    # L_r: reward prediction
    loss_r = F.mse_loss(reward_head(z).squeeze(-1), r)

    # L_p: next-representation prediction against the EMA target encoder and projection
    z_hat = trans_pred(torch.cat([z, F.one_hot(a, num_classes=n_a).float(), y], dim=-1))
    h_hat = F.normalize(pred_online(proj_online(z_hat)), p=2, dim=1)
    with torch.no_grad():
        h_tgt = F.normalize(proj_target(target_model(nctx, nmsk)), p=2, dim=1)
    loss_p = F.mse_loss(h_hat, h_tgt)

    # L_SI: Soft-InfoNCE with graded targets from the context similarity
    g = F.normalize(proj_online(z), p=2, dim=1)
    eye = torch.eye(Bsz, device=device, dtype=torch.bool)
    logits = ((g @ g.t()) / SI_TEMPERATURE).masked_fill(eye, -1e9)
    s = Context_Similarity(th).masked_fill(eye, 0.0)
    w = s / s.sum(dim=1, keepdim=True).clamp(min=1e-8)
    loss_si = -(w * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

    loss = LAMBDA_R * loss_r + LAMBDA_P * loss_p + LAMBDA_THETA * loss_si

    encoder_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(Context_Model.parameters()) + list(trans_pred.parameters()) +
        list(reward_head.parameters())  + list(proj_online.parameters()) +
        list(pred_online.parameters()), max_norm=1.0)
    encoder_optimizer.step()
    ema_update()

    Context_Model.eval()
    return loss.item()

## Game step: critic regression and actor distillation

In [ ]:
def unroll_zs(ctx, msk):
    # Recompute z^S by running the state GRU over the stored history rows
    z = torch.zeros(ctx.shape[0], Z_S_Size, device=device)
    T = int(msk.sum(dim=1).max().item())
    for t in range(T):
        step = msk[:, t].float().unsqueeze(1)
        z = step * Z_S_ENC(z, ctx[:, t, N:2*N], ctx[:, t, 0:N]) + (1.0 - step) * z
    return z
#################################################################################################################
def _game_batch(batch_size):
    # Sample transitions; recompute both representation components with the current encoders
    batch = exp_memory.sample(batch_size)
    (x, nx, a, u, r, tt, th, m2, nm2, ctx, msk, nctx, nmsk) = map(list, zip(*batch))
    S = lambda v: torch.stack(v).to(device)
    C = lambda v: torch.cat(v, dim=0).to(device)
    ctx, msk, nctx, nmsk = C(ctx), C(msk), C(nctx), C(nmsk)
    Context_Model.eval()
    with torch.no_grad():
        emb, nemb = Context_Model(ctx, msk), Context_Model(nctx, nmsk)
        zs, nzs = unroll_zs(ctx, msk), unroll_zs(nctx, nmsk)
    return dict(x=S(x), nx=S(nx), a=S(a), u=S(u), r=S(r), tt=S(tt), th=S(th), m2=S(m2), nm2=S(nm2),
                ctx=ctx, msk=msk, emb=emb, nemb=nemb, zs=zs, nzs=nzs)
#################################################################################################################
def Train_Critic():
    if len(exp_memory) < CRITIC_BATCH:
        return 0.0
    b = _game_batch(CRITIC_BATCH)
    Bsz = b['x'].shape[0]

    q_flat   = critic(b['zs'], b['emb'], b['x'], b['m2'], b['tt'], b['th'])
    q_values = q_flat.gather(1, (b['a'] * n_u + b['u']).unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_q = critic_target(b['nzs'], b['nemb'], b['nx'], b['nm2'], b['tt'], b['th']).view(Bsz, n_a, n_u)
        V_next = batched_game_values(next_q, legal_mask_from_theta(b['th']), iters=200)
        target_values = b['r'] + GAMMA * V_next

    loss = F.smooth_l1_loss(q_values, target_values)
    critic_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(critic.parameters(), max_norm=1.0)
    critic_optimizer.step()

    with torch.no_grad():
        for tgt, src in zip(critic_target.parameters(), critic.parameters()):
            tgt.data.mul_(1.0 - TAU).add_(TAU * src.data)
    return float(loss.item())
#################################################################################################################
def _equilibrium_targets(b, which):
    # Exact stage-game equilibrium over the admissible attacker actions: which = 1 row, which = 2 column
    Bsz = b['x'].shape[0]
    with torch.no_grad():
        Q = critic(b['zs'], b['emb'], b['x'], b['m2'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
    legal = legal_mask_from_theta(b['th']).cpu().numpy()
    tgt = np.zeros((Bsz, n_a if which == 1 else n_u), dtype=np.float32)
    for i in range(Bsz):
        cols = np.where(legal[i])[0]
        _, p, q = MatrixValue(Q[i][:, cols])
        if which == 1:
            tgt[i] = p
        else:
            tgt[i, cols] = q
    return torch.tensor(tgt, device=device)
#################################################################################################################
def Train_Player1():
    # Defender (and its state GRU): cross-entropy toward the row (maximizing) equilibrium strategy
    if len(exp_memory) < PLAYER1_BATCH:
        return 0.0
    b = _game_batch(PLAYER1_BATCH)
    tgt_a = _equilibrium_targets(b, 1)
    zs_live = unroll_zs(b['ctx'], b['msk'])                 # with gradient, so the state GRU is trained too
    loss = -(tgt_a * F.log_softmax(Player1(zs_live, b['emb']), dim=1)).sum(dim=1).mean()

    player1_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(list(Player1.parameters()) + list(Z_S_ENC.parameters()), max_norm=1.0)
    player1_optimizer.step()
    return loss.item()
#################################################################################################################
def Train_Player2():
    # Attacker: cross-entropy toward the column (minimizing) equilibrium strategy over its admissible actions
    if len(exp_memory) < PLAYER2_BATCH:
        return 0.0
    b = _game_batch(PLAYER2_BATCH)
    tgt_u = _equilibrium_targets(b, 2)
    pred_u = Player2(b['x'], b['m2'], b['tt'])
    loss = -(tgt_u * torch.log(pred_u + 1e-8)).sum(dim=1).mean()

    player2_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(Player2.parameters(), max_norm=1.0)
    player2_optimizer.step()
    return loss.item()

## Baseline architectures and the best-response attacker (shared by baselines, best_response, and eval)

In [ ]:
OBS_DIM = 2 * N        # baseline observation: [y, a_prev / B]
RNN_H = 64
LATENT_DIM = 16


class RecurrentAC(nn.Module):
    # Recurrent actor-critic over the baseline observation, optionally conditioned on an extra input
    def __init__(self, extra_dim=0, rnn_h=RNN_H):
        super().__init__()
        self.extra_dim = extra_dim
        self.gru = nn.GRU(OBS_DIM, rnn_h, batch_first=True)
        dd = rnn_h + extra_dim
        self.pi = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, n_a))
        self.v  = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, obs, extra=None, h0=None):
        out, hT = self.gru(obs, h0)
        feat = out if extra is None else torch.cat([out, extra], dim=-1)
        return self.pi(feat), self.v(feat).squeeze(-1), hT
#################################################################################################################
class VariBADEncoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, rnn_h=RNN_H):
        super().__init__()
        self.gru = nn.GRU(OBS_DIM, rnn_h, batch_first=True)
        self.mu = nn.Linear(rnn_h, latent_dim)
        self.logvar = nn.Linear(rnn_h, latent_dim)
    def forward(self, obs, h0=None):
        out, hT = self.gru(obs, h0)
        return self.mu(out), self.logvar(out), hT
#################################################################################################################
class PEARLEncoder(nn.Module):
    # Product-of-Gaussians posterior over the latent from the episode's transitions (y, a / B, y_next)
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(nn.Linear(3 * N, 128), nn.ReLU(), nn.Linear(128, 2 * latent_dim))
    def factors(self, trans):
        out = self.net(trans)
        mu = out[..., :self.latent_dim]
        var = F.softplus(out[..., self.latent_dim:]) + 1e-6
        return mu, var
    def posterior(self, trans):
        if trans.shape[0] == 0:
            return torch.zeros(self.latent_dim, device=device), torch.ones(self.latent_dim, device=device)
        mu_i, var_i = self.factors(trans)
        prec = 1.0 / var_i
        prec_post = 1.0 + prec.sum(0)
        mu_post = (prec * mu_i).sum(0) / prec_post
        return mu_post, 1.0 / prec_post
#################################################################################################################
class BRAttacker(nn.Module):
    # Recurrent best-response attacker: GRU over [true state, previous allocation / B], conditioned on the full theta,
    # masked to the admissible allocations
    def __init__(self, rnn_h=RNN_H):
        super().__init__()
        self.gru = nn.GRU(2 * N, rnn_h, batch_first=True)
        self.theta_enc = nn.Sequential(nn.Linear(Len_Theta, 64), nn.ReLU())
        dd = rnn_h + 64
        self.pi = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, n_u))
        self.v  = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, obs, theta, legal_mask, h0=None):
        out, hT = self.gru(obs, h0)
        feat = torch.cat([out, self.theta_enc(theta)], dim=-1)
        logits = self.pi(feat).masked_fill(~legal_mask, -1e9)
        return logits, self.v(feat).squeeze(-1), hT

## Loaders and defender wrappers (shared by best_response, eval, and control)

In [ ]:
def Load(net, path):
    net.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    return net.eval()


def Load_Baselines(save_dir="compared"):
    # The five trained baselines from baselines.ipynb
    ck_ = lambda name: torch.load(os.path.join(save_dir, name + ".pth"), map_location=device, weights_only=True)
    out = {}
    for name in ('RPPO', 'DR_RPPO'):
        ck = ck_(name)
        pol = RecurrentAC(0, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['policy']); pol.eval()
        out[name] = {'policy': pol}
    ck = ck_("VARIBAD"); L = ck['config']['LATENT_DIM']
    pol = RecurrentAC(2 * L, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['policy']); pol.eval()
    enc = VariBADEncoder(L, ck['config']['RNN_H']).to(device); enc.load_state_dict(ck['encoder']); enc.eval()
    out['VARIBAD'] = {'policy': pol, 'encoder': enc, 'latent_dim': L}
    ck = ck_("PEARL"); L = ck['config']['LATENT_DIM']
    pol = RecurrentAC(L, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['policy']); pol.eval()
    enc = PEARLEncoder(L).to(device); enc.load_state_dict(ck['encoder']); enc.eval()
    out['PEARL'] = {'policy': pol, 'encoder': enc, 'latent_dim': L, 'resample_k': ck['config'].get('resample_k', 10)}
    ck = ck_("MMD")
    pol = RecurrentAC(0, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['p1']); pol.eval()
    out['MMD'] = {'policy': pol}
    return out
#################################################################################################################
# Defender wrappers: reset(Theta, Type_Theta, legal) at the start of an episode, act_dist(x) for the action
# distribution, observe(x, a_vec, u_vec, next_x, y) after each transition. They use the globals Context_Model,
# Z_S_ENC, critic, Player1, and BASELINES, which each evaluation notebook loads.
class PipelineDefender:
    # 'Trained P1': our defender on [z^S, z^Theta]. 'Critic P1': the Oracle, the critic's equilibrium row
    # with privileged access to the state and context.
    def __init__(self, name):
        self.name = name
    def reset(self, Theta, Type_Theta, legal):
        self.th = torch.tensor(Theta, dtype=torch.float32, device=device)
        self.tt = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        self.legal = legal
        self.z_s = torch.zeros(Z_S_Size, dtype=torch.float32, device=device)
        self.Context = np.zeros([max_steps, N * 2], dtype=np.float32)
        self.m2 = np.full(N * 2, 0.5)
        self.t = 0
    def act_dist(self, x):
        ctx = torch.tensor(self.Context, dtype=torch.float32, device=device).view(1, max_steps, N * 2)
        with torch.no_grad():
            Emb = Context_Model(ctx, Context_Mask(self.t)).squeeze(0)
            if self.name == 'Trained P1':
                p = F.softmax(Player1(self.z_s, Emb), dim=-1).cpu().numpy()
            else:
                Q = critic(self.z_s.unsqueeze(0), Emb.unsqueeze(0),
                           torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0),
                           torch.tensor(self.m2, dtype=torch.float32, device=device).unsqueeze(0),
                           self.tt.unsqueeze(0), self.th.unsqueeze(0)).squeeze(0).cpu().numpy().reshape(n_a, n_u)
                _, p, _ = MatrixValue(Q[:, self.legal])
        p = np.clip(np.asarray(p, dtype=np.float64), 0, None)
        return p / p.sum()
    def observe(self, x, a_vec, u_vec, next_x, y):
        if self.name == 'Critic P1':
            self.m2 = m2_update(self.m2, x, u_vec, next_x)
        with torch.no_grad():
            self.z_s = Z_S_ENC(self.z_s, torch.tensor(y, dtype=torch.float32, device=device),
                               torch.tensor(a_vec, dtype=torch.float32, device=device))
        self.Context[self.t, :] = Context_Row(a_vec, y)
        self.t += 1


class RandomDefender:
    def reset(self, Theta, Type_Theta, legal):
        pass
    def act_dist(self, x):
        return np.full(n_a, 1.0 / n_a)
    def observe(self, x, a_vec, u_vec, next_x, y):
        pass


class BaselineDefender:
    # A trained baseline acting on [y, a_prev / B], with VariBAD's belief or PEARL's sampled latent as extra input
    def __init__(self, name):
        self.name = name
        self.m = BASELINES[name]
    def reset(self, Theta, Type_Theta, legal):
        self.h = None
        self.y_cur = np.zeros(N)
        self.a_prev = np.zeros(N)
        if self.name == 'VARIBAD':
            self.enc_h = None
        if self.name == 'PEARL':
            self.ctx = []
            self.z = np.random.randn(self.m['latent_dim']).astype(np.float32)
    def _extra(self, obs):
        if self.name == 'VARIBAD':
            with torch.no_grad():
                mu, logvar, self.enc_h = self.m['encoder'](
                    torch.tensor(obs, dtype=torch.float32, device=device).view(1, 1, -1), self.enc_h)
            mu = mu.view(-1).cpu().numpy()
            std = np.exp(0.5 * logvar.view(-1).cpu().numpy())
            return np.concatenate([mu, std])
        if self.name == 'PEARL':
            return self.z
        return None
    def act_dist(self, x):
        obs = np.concatenate([self.y_cur, self.a_prev]).astype(np.float32)
        ex_np = self._extra(obs)
        obs_t = torch.tensor(obs, device=device).view(1, 1, OBS_DIM)
        ex_t = None if ex_np is None else torch.tensor(np.asarray(ex_np, dtype=np.float32), device=device).view(1, 1, -1)
        with torch.no_grad():
            logits, _, self.h = self.m['policy'](obs_t, ex_t, self.h)
        p = F.softmax(logits.view(-1), dim=-1).cpu().numpy().astype(np.float64)
        return p / p.sum()
    def observe(self, x, a_vec, u_vec, next_x, y):
        if self.name == 'PEARL':
            self.ctx.append(np.concatenate([self.y_cur, a_vec / B, y]).astype(np.float32))
            if len(self.ctx) % self.m['resample_k'] == 0:
                with torch.no_grad():
                    mu, var = self.m['encoder'].posterior(torch.tensor(np.stack(self.ctx), device=device))
                self.z = (mu + var.sqrt() * torch.randn_like(mu)).cpu().numpy()
        self.y_cur = y
        self.a_prev = a_vec / B


def get_defender(name):
    if name in ('Critic P1', 'Trained P1'):
        return PipelineDefender(name)
    if name == 'Random P1':
        return RandomDefender()
    return BaselineDefender(name)